# Verbatim, time-aligned transcripts: CrisperWhisper 2.0 + MFA v3

Runs the pipeline in `scripts/` end to end on a Colab GPU runtime
(**Runtime -> Change runtime type -> GPU**). Every step skips work that is already
on Drive, so after a disconnect you can just re-run from the top.

| step | script | output |
|---|---|---|
| 1 | `01_prepare_audio.py` | `data/wav/*.wav`, `data/manifest.csv` |
| 2 | `02_transcribe_crisperwhisper.py` | `data/transcripts/crisperwhisper/*.json` |
| 3 | `03_prepare_mfa_corpus.py` | `data/mfa_corpus/<participant>/*.{wav,TextGrid}` |
| 4 | `04_run_mfa.sh` | `data/mfa_aligned/**/*.TextGrid` |
| 5 | `05_merge_alignments.py` | `data/aligned/` (tokens, pauses, TextGrids, summary) |

## 0. Setup

In [ ]:
# Where the repo lives on Drive (upload the folder, or git clone it there).
REPO_DIR = "/content/drive/MyDrive/cognitive_load"

from google.colab import drive
drive.mount("/content/drive")
%cd {REPO_DIR}
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
# Keep the venv, HF cache and MFA environment on fast local disk (they are rebuilt per session).
os.environ["UV_PROJECT_ENVIRONMENT"] = "/content/.venv"
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["MFA_ENV"] = "/content/mfa_env"
os.environ["MFA_ROOT_DIR"] = "/content/mfa_root"
PY = "/content/.venv/bin/python"

# CrisperWhisper weights are under a non-commercial research licence. If the HF repo
# asks you to accept terms, do so on the model page and add an HF_TOKEN Colab secret.
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("no HF_TOKEN secret set (fine if the model is not gated)")

!pip install -q uv
!uv sync --extra ct2

## 1. Convert audio to 16 kHz mono WAV + manifest
Check the printed IDs: participant / day / prompt / condition are parsed from the file names.

In [ ]:
!{PY} scripts/01_prepare_audio.py --audio-root audio/LCC_Data --out-dir data/wav --manifest data/manifest.csv

## 2. CrisperWhisper 2.0 (verbatim, word timestamps)
Smoke-test on one file first. The first run downloads the model and converts it to
CTranslate2; `--ct2-cache` on Drive means the conversion only happens once.
Add `--dual` to also store the clean "intended" transcript (handy later for counting ideas).

In [ ]:
CW_ARGS = "--wav-dir data/wav --out-dir data/transcripts/crisperwhisper --model large --ct2-cache /content/drive/MyDrive/cognitive_load_cache/ct2"
!{PY} scripts/02_transcribe_crisperwhisper.py {CW_ARGS} --limit 1
!head -c 1500 data/transcripts/crisperwhisper/*.txt

In [ ]:
!{PY} scripts/02_transcribe_crisperwhisper.py {CW_ARGS}

## 3. Build the MFA corpus (chunk at pauses, normalise fillers/partials)

In [ ]:
!{PY} scripts/03_prepare_mfa_corpus.py --transcripts data/transcripts/crisperwhisper --out-dir data/mfa_corpus

## 4. Montreal Forced Aligner v3
First run installs micromamba + MFA from conda-forge into `/content/mfa_env` (a few minutes).
Uses the `english_mfa` acoustic model with the **Indian English** dictionary and G2P.

In [ ]:
!bash scripts/04_run_mfa.sh data/mfa_corpus data/mfa_aligned

## 5. Merge MFA timings onto the verbatim tokens; extract pauses

In [ ]:
!{PY} scripts/05_merge_alignments.py --transcripts data/transcripts/crisperwhisper --mfa-dir data/mfa_aligned --out-dir data/aligned --min-pause 0.25

In [ ]:
import pandas as pd
summary = pd.read_csv("data/aligned/summary.csv")
display(summary.sort_values("mfa_coverage").head(10))
display(summary.groupby("condition")[["wpm", "fillers_per_min", "n_pauses", "mean_pause_s"]].mean())